# Goodreads Book Scraper

This notebook scrapes book data from Goodreads for books related to Python and saves the results to a CSV file.

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

## Define Headers and Initialize Variables
Set up the HTTP headers and lists to store book data.

In [ ]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0.0.0 Safari/537.36"
    )
}

title = []
author = []
avg_rating = []
rating = []
year_published = []
book_links = []

## Set Maximum Pages and Books
Specify how many pages and books to scrape.

In [ ]:
max_pages = 100
books_per_page = 10
max_books = 1000

## Scrape Goodreads Data
Iterate through the pages, send HTTP requests, parse the HTML, and extract book details.

In [ ]:
title = []
author = []
avg_rating = []
rating = []
year_published = []
book_links = []

for page in range(1, max_pages + 1):
    url = f"https://www.goodreads.com/search?page={page}&q=Python&qid=eVVSpjWBj5&tab=books"
    print(f"Fetching page {page}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to retrieve page {page}. Status code: {response.status_code}")
        break

    soup = BeautifulSoup(response.text, "lxml")

    containers = soup.find_all("tr")

    if not containers:
        print(f"No book containers found on page {page}.")
        break
    for book in containers:
        try:
            book_title = book.find("span").text
            book_author = book.find("a", class_ = "authorName").text
            rating_text = book.find("span", class_="minirating").text.strip()
            average_rating = rating_text.split(" avg rating")[0]
            ratings = rating_text.split("—")[-1].split("ratings")[0].strip()
            year_span = book.find("span", class_ = "greyText smallText uitext")
            published_year = "Not found"

            if year_span:
                year_text = year_span.text
                if "published" in year_text:
                    parts = year_text.split("published")
                    if len(parts) > 1:
                        published_year = parts[1].strip().split()[0]

            book_link = book.find("a", class_ = "bookTitle")['href']   
            full_link = "https://www.goodreads.com/" + book_link

            title.append(book_title)
            author.append(book_author)
            avg_rating.append(average_rating)
            rating.append(ratings)
            year_published.append(published_year)
            book_links.append(full_link)

            if len(title) >= max_books:
                print(f"Reached the max number of books: {max_books}. Stopping......")
                break

        except Exception as e:
            print(f"Error processing book: {e}\nSkipping to the next book...")
            continue    

    if len(title) >= max_books:
        break    

    time.sleep(0.5)  # Sleep to avoid overwhelming the server

## Create DataFrame and Save to CSV
Convert the scraped data into a pandas DataFrame and save it as a CSV file.

In [ ]:
df = pd.DataFrame({
    "Title": title,
    "Author": author,
    "Avg_rating": avg_rating,
    "Rating": rating,
    "Published_year": year_published,
    "links": book_links
})

print(f"\nScraped {len(df)} books from Goodreads.")

df.to_csv("goodreads_scraped.csv", index=False)
print("Data saved to goodreads_scraped.csv")